# asyncio — Core Tools

Beyond basic async/await, asyncio gives you tools to
manage, control, and handle multiple tasks properly.

# asyncio.gather() - with error handling

In [2]:
import asyncio

async def risky_task(name, should_fail=False):
    await asyncio.sleep(1)
    if should_fail:
        raise ValueError(f"{name} failed")
    return  f"{name} succeeded"

# Without return exception - one failure crashes all

In [3]:
try:
    results = await asyncio.gather(
        risky_task(name="A"),
        risky_task(name="B", should_fail=True),
        risky_task(name="C"),
    )
except ValueError as e:
    print(f"Everything stopped because: {e}")

Everything stopped because: B failed


# With return_exceptions - safe for agents

In [4]:
results = await asyncio.gather(
    risky_task(name="A"),
    risky_task(name="B", should_fail=True),
    risky_task(name="C"),
    return_exceptions=True
)

for result in results:
    if isinstance(result, Exception):
        print(f"Task failed: {result}")
    else:
        print(f"Task ok: {result}")

Task ok: A succeeded
Task failed: B failed
Task ok: C succeeded


# asyncio.create_task() - fire and don't wait

In [5]:
async def background_logger():
    for i in range(3):
        print(f"Background log {i}")
        await asyncio.sleep(1)

async def main_work():
    task = asyncio.create_task(background_logger())

    print("Doing main work")
    await asyncio.sleep(1)
    print("Main work done")

    await task

await main_work()

Doing main work
Background log 0
Main work done
Background log 1
Background log 2


# Timeout - critical for agents

In [6]:
async def slow_tool():
    await asyncio.sleep(10)
    return "finally done"

try:
    result = await asyncio.wait_for(slow_tool(), timeout=3)
except asyncio.TimeoutError:
    print("Tool timed out after 3 seconds, agent can handle this gracefully")

Tool timed out after 3 seconds, agent can handle this gracefully


# asyncio.sleep() vs time.sleep()
NEVER use time.sleep() in async code.

time.sleep(2)      → freezes the ENTIRE program for 2 seconds

asyncio.sleep(2)   → pauses only THIS coroutine, others keep running